# Quantum sensing with NV centres

A nitrogen-vacancy (NV) centre in diamond is an electron spin 1 that can
be prepared and read out with light at room temperature. Its levels
`m_s = ±1` shift with the magnetic field, so the centre is a magnetometer
of nanometre size; it senses fields inside and next to living cells (Le
Sage et al., Nature 496, 486 (2013)) and single nuclear spins in the
diamond. What it can sense is decided by its coherence, and pulse
sequences that refocus the electron (echoes, dynamical decoupling) both
protect that coherence and make the centre listen at chosen frequencies.

This notebook builds a centre with its ¹⁴N nucleus, finds its lines,
watches the nitrogen in Ramsey fringes, detects a single ¹³C nucleus with
CPMG against the closed form of Taminiau et al. (Phys. Rev. Lett. 109,
137602 (2012)), compares CPMG and XY-8 with finite pulses, and measures
how decoupling fights classical field noise, against the filter-function
theory and exact noisy trajectories.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from metatron_qso import spin

D, GAMMA_E = spin.NV_ZERO_FIELD_SPLITTING, spin.NV_GYROMAGNETIC_RATIO  # Hz, Hz/T
B = 0.0403  # T along the NV axis

## Lines and pulsed ODMR

In the secular approximation the electron levels are `D m_s² + γ_e B m_s`
and the ¹⁴N hyperfine coupling `A_N = −2.14 MHz` splits each transition
into three lines, one per nuclear level `m_I`. `lines()` lists them with
their strengths. Pulsed optically detected magnetic resonance (ODMR)
applies a π pulse at each microwave frequency and reads the population of
`m_s = 0`: a weak pulse flips only the nuclear third that is on
resonance, so each line dips to 2/3.

In [ ]:
nv = spin.NvCenter(B, nitrogen="14N")
for f, strength in nv.lines():
    print(f"{f / 1e6:10.3f} MHz   strength {strength:.3f}")

centre = D - GAMMA_E * B
freqs = centre + np.linspace(-4e6, 4e6, 401)
rabi = 0.3e6
odmr = nv.odmr(freqs, rabi, 1 / (2 * rabi))
plt.plot((freqs - centre) / 1e6, odmr)
plt.xlabel("microwave − (D − γ_e B)  (MHz)")
plt.ylabel("population of m_s = 0")
plt.show()

## Ramsey fringes

A Ramsey sequence `π/2 − τ − π/2` measures the phase the coherence
gathers while the microwave is detuned. Each nuclear level contributes
its own detuning `δ + A_N m_I`, so the fringes beat; with ideal pulses
the signal is `½ + ⅙ Σ_m cos 2π(δ + A_N m)τ` exactly.

In [ ]:
delta = 1e6  # Hz above the bare line
taus = np.linspace(0, 3e-6, 301)
ramsey = np.array([nv.signal(spin.ramsey(t), frequency=centre + delta) for t in taus])
closed = 0.5 + sum(np.cos(2 * np.pi * (delta - 2.14e6 * m) * taus) for m in (-1, 0, 1)) / 6
print(f"largest deviation from the closed form: {np.abs(ramsey - closed).max():.1e}")
plt.plot(taus * 1e6, ramsey)
plt.xlabel("τ (μs)")
plt.ylabel("population of m_s = 0")
plt.show()

## Detecting one ¹³C nucleus

A nearby ¹³C nucleus precesses at its Larmor frequency, slightly
differently for each electron level because of its hyperfine coupling.
When the π pulses of CPMG are spaced in step with that precession, the
nucleus is rotated conditionally on the electron, and the electron loses
coherence. The signal shows dips at
`τ_k ≈ (2k − 1)π/(2ω_L + A_∥)`. Taminiau et al. give it in closed form;
the ¹⁴N shift is refocused by the ideal π pulses.

In [ ]:
a_par, a_perp = 60e3, 30e3  # Hz
nv_c = spin.NvCenter(B, nitrogen="14N", carbons=[(a_par, a_perp)])
omega_l = -spin.gyromagnetic_ratio("13C") * B  # rad/s
a, b = -2 * np.pi * a_par, -2 * np.pi * a_perp


def taminiau(tau, n):
    w = np.hypot(a + omega_l, b)
    mz, mx = (a + omega_l) / w, b / w
    alpha, beta = w * tau, omega_l * tau
    cos_phi = np.cos(alpha) * np.cos(beta) - mz * np.sin(alpha) * np.sin(beta)
    phi = np.arccos(np.clip(cos_phi, -1, 1))
    m = 1 - mx**2 * (1 - np.cos(alpha)) * (1 - np.cos(beta)) / (1 + cos_phi) * np.sin(n * phi / 2) ** 2
    return (1 + m) / 2


taus = np.linspace(0.4e-6, 3.2e-6, 1121)
cpmg = np.array([nv_c.signal(spin.cpmg(64, t)) for t in taus])
print(f"largest deviation from Taminiau et al.: {np.abs(cpmg - taminiau(taus, 64)).max():.1e}")
plt.plot(taus * 1e6, cpmg)
for k in range(1, 4):
    plt.axvline((2 * k - 1) * np.pi / (2 * abs(omega_l) + 2 * np.pi * a_par) * 1e6, color="gray", lw=0.5)
plt.xlabel("τ (μs), π pulses spaced 2τ")
plt.ylabel("population of m_s = 0")
plt.show()

## Finite pulses: CPMG against XY-8

Real pulses take time. With a Rabi frequency of 20 MHz a π pulse lasts
25 ns, and it is off resonance by ±2.14 MHz for two of the three ¹⁴N
levels. CPMG corrects such errors only for coherence along its pulse
axis; the electron–carbon terms that build up at a resonance are not, and
64 pulses wash the dip out. XY-8 alternates the pulse axes and keeps it.

In [ ]:
near = np.linspace(0.50e-6, 0.58e-6, 161)
curves = {
    "CPMG, ideal pulses": [nv_c.signal(spin.cpmg(64, t)) for t in near],
    "CPMG, 20 MHz": [nv_c.signal(spin.cpmg(64, t), rabi=20e6) for t in near],
    "XY-8, 20 MHz": [nv_c.signal(spin.xy8(8, t), rabi=20e6) for t in near],
}
for label, values in curves.items():
    print(f"{label:20s} deepest {min(values):.3f}")
    plt.plot(near * 1e6, values, label=label)
plt.xlabel("τ (μs)")
plt.ylabel("population of m_s = 0")
plt.legend()
plt.show()

## Decoupling against field noise

A fluctuating field `b(t)` along the axis adds a random phase
`φ = 2π∫ b(t) y(t) dt`, where `y = ±1` flips at every π pulse. For
Gaussian noise of spectrum `S(ω)` the coherence decays as `e^{−χ}` with
`χ = 2π∫₀^∞ S(ω)|Y(ω)|² dω` (Cywiński et al., Phys. Rev. B 77, 174509
(2008)); the filter function `|Y(ω)|²` of a sequence decides which
frequencies of the noise it lets through. Here Ornstein–Uhlenbeck noise
of 200 kHz and 1 μs correlation time is compared with exact trajectories.

In [ ]:
noise = spin.FieldNoise.ornstein_uhlenbeck(2e5, 1e-6)
bare = spin.NvCenter(B)
totals = np.linspace(0.2e-6, 6e-6, 30)
sequences = {
    "Ramsey": lambda T: spin.ramsey(T),
    "Hahn echo": lambda T: spin.hahn(T / 2),
    "CPMG-8": lambda T: spin.cpmg(8, T / 16),
}
for label, make in sequences.items():
    chi = np.array([spin.decay_exponent(make(T), noise) for T in totals])
    line = plt.plot(totals * 1e6, np.exp(-chi), label=label)[0]
    sample = totals[::4]
    runs = [bare.signal_with_noise(make(T), noise, 2000, seed=k) for k, T in enumerate(sample)]
    coherence = [2 * m - 1 for m, _ in runs]
    errors = [2 * e for _, e in runs]
    plt.errorbar(sample * 1e6, coherence, yerr=errors, fmt="o", color=line.get_color())
plt.xlabel("total free time (μs)")
plt.ylabel("coherence")
plt.legend()
plt.show()

The filter functions show why: at the same total time, more π pulses move
the window of `|Y(ω)|²` to higher frequencies, where this noise has less
power.

In [ ]:
T = 4e-6
omega = np.linspace(1e3, 2.5e7, 2000)
for label, make in sequences.items():
    plt.plot(omega / (2 * np.pi * 1e6), spin.filter_function(make(T), omega) * 1e12, label=label)
plt.plot(omega / (2 * np.pi * 1e6), noise.spectrum(omega) / noise.spectrum([0.0])[0] * 4, "k--", label="S(ω), scaled")
plt.xlabel("noise frequency (MHz)")
plt.ylabel("|Y(ω)|² (μs²)")
plt.legend()
plt.show()

## When noise is not Gaussian

A single fluctuator that jumps between `±a` (telegraph noise) is not
Gaussian. When it flips slowly compared with `2πa`, the coherence
oscillates instead of decaying, and `e^{−χ}`, which only knows the
spectrum, is wrong. The exact Ramsey coherence is Kubo and Anderson's
`e^{−γt}(cos ωt + (γ/ω) sin ωt)`, `ω = √((2πa)² − γ²)`, with `γ` the
flip rate; the trajectories follow it.

In [ ]:
amplitude, rate = 3e5, 2e5
telegraph = spin.FieldNoise.telegraph(amplitude, rate)
times = np.linspace(0.05e-6, 4e-6, 40)
nu = 2 * np.pi * amplitude
w = math.sqrt(nu**2 - rate**2)
exact = np.exp(-rate * times) * (np.cos(w * times) + rate / w * np.sin(w * times))
gaussian = np.exp(-np.array([spin.decay_exponent(spin.ramsey(t), telegraph) for t in times]))
runs = [bare.signal_with_noise(spin.ramsey(t), telegraph, 4000, seed=k) for k, t in enumerate(times)]
plt.plot(times * 1e6, exact, label="Kubo–Anderson")
plt.plot(times * 1e6, gaussian, "--", label="e^{−χ} (Gaussian)")
plt.errorbar(times * 1e6, [2 * m - 1 for m, _ in runs], yerr=[2 * e for _, e in runs], fmt="o", ms=3, label="trajectories")
plt.xlabel("τ (μs)")
plt.ylabel("Ramsey coherence")
plt.legend()
plt.show()
worst = max(abs(2 * m - 1 - x) / (2 * e) for (m, e), x in zip(runs, exact))
print(f"largest deviation of the trajectories from Kubo–Anderson: {worst:.1f} standard errors")

The trajectories are exact for every noise QSO describes: between pulses
the electron's `S_z` commutes with the Hamiltonian, so a noise path only
adds the phase of its integral over each period, which is sampled
exactly. With finite pulses the noise is also applied during the pulses.